In [605]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 120652 bytes to preprocess.ipynb


In [708]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pulp import *
from scipy.stats import poisson, norm
from pulp import LpProblem, LpMaximize, LpVariable, LpStatus, lpSum, value, PULP_CBC_CMD

In [709]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [710]:
df= pd.read_csv("fantasy_enriched.csv")

In [711]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [712]:
df = df[df["status"] == "playing"].copy()

In [713]:
# probability of playing a minute

raw_any = (
    0.2* df["minutes_rank_team_position"]
  + 0.10 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.15 * df["starts_rank_team_position"]
  + 0.10 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.1 * df["selected_rank_team_position"]
  + 0.25 * df["minutes_last_3_avg"]
  + 0.1 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6)

In [714]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
10,0.937027,Emiliano Martínez,Argentina,GK,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000
211,0.931775,Gustavo Gómez,Paraguay,DEF,1.000000,1.000000,0.571429,1.000000,1.000000,1.000000
394,0.931263,Achraf Hakimi,Morocco,DEF,1.000000,1.000000,0.562500,1.000000,1.000000,1.000000
148,0.920195,Kylian Mbappé,France,FWD,0.900000,1.000000,0.666667,1.000000,1.000000,1.000000
127,0.918872,Harry Kane,England,FWD,0.907692,1.000000,0.625000,1.000000,1.000000,1.000000
86,0.915806,Luis Díaz,Colombia,MID,0.917949,1.000000,0.545455,1.000000,1.000000,1.000000
350,0.915676,Cristiano Ronaldo,Portugal,FWD,0.900000,1.000000,0.600000,1.000000,1.000000,1.000000
65,0.915617,Jonathan David,Canada,FWD,0.848718,1.000000,0.583333,1.000000,1.000000,1.000000
285,0.913635,Breel Embolo,Switzerland,FWD,0.889744,1.000000,0.600000,1.000000,1.000000,1.000000
48,0.912136,Vinícius Júnior,Brazil,MID,0.900000,1.000000,0.550000,1.000000,1.000000,1.000000


In [715]:
# probability of starting / playing 60 minutes

raw_any = (
    0.15 * df["minutes_rank_team_position"]
  + 0.10 * df["starts_rank_team_position"]
  + 0.15 * df["stat_MP"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.35 * df["minutes_last_3_avg"]
  + 0.15 * df["round_4_MP"]
  + 0.05 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [716]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_4_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,name,team,position,prob_plays_60min,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_4_MP,anytime_rank_team_position
10,Emiliano Martínez,Argentina,GK,0.942676,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,1.000000
211,Gustavo Gómez,Paraguay,DEF,0.939509,1.000000,0.571429,1.000000,1.000000,1.000000,1.000000,1.000000
394,Achraf Hakimi,Morocco,DEF,0.939204,1.000000,0.562500,1.000000,1.000000,1.000000,1.000000,1.000000
107,Mohamed Hany,Egypt,DEF,0.936805,1.000000,0.562500,0.500000,0.862500,1.000000,1.000000,0.500000
28,Brandon Mechele,Belgium,DEF,0.934625,1.000000,0.555556,0.277778,0.755556,1.000000,1.000000,0.777778
39,Youri Tielemans,Belgium,MID,0.930263,1.000000,0.545455,0.727273,0.700000,0.983333,1.000000,0.500000
401,Neil El Aynaoui,Morocco,MID,0.928121,1.000000,0.555556,0.444444,0.633333,0.976667,1.000000,0.555556
360,Andrés Cubas,Paraguay,MID,0.927574,1.000000,0.555556,0.277778,0.388889,1.000000,1.000000,0.222222
340,Thibaut Courtois,Belgium,GK,0.924142,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,0.666667
227,Orlando Gill,Paraguay,GK,0.924142,1.000000,0.666667,0.333333,1.000000,1.000000,1.000000,0.666667


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [ ]:
df["lambda_goal"] = -np.log(
    (1 - df["anytime_scorer_prob"]).clip(lower=0.001)  # clip prevents log(0)
)

lambda_adj = (
    0.80 * df["lambda_goal"]                          # dominant: Betano's model
      + 0.05 * df["recent_goals_last3"]
      + 0.05 * df["team_score_2_prob"]
      + 0.02 * df["stat_GS"]
      + 0.03 * df["stat_ST"]
      + 0.05 * df["price"]
  )

# expected_goals is λ (not a probability — can exceed 1 for elite strikers)
df["prob_scores"] = lambda_adj.clip(lower=0) * df["prob_plays_60min"] * 1.1

In [755]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
148,Kylian Mbappé,France,FWD,0.924373,0.558959,0.6536,1.00,0.034483,0.324786,0.351852,0.857143,0.866667,1.000000
5,Lionel Messi,Argentina,FWD,0.852061,0.499396,0.6173,1.00,0.045833,0.415625,0.445312,1.000000,1.000000,1.000000
48,Vinícius Júnior,Brazil,MID,0.585635,0.369341,0.4717,0.75,0.034483,0.216524,0.270655,0.571429,0.666667,1.000000
261,Mikel Oyarzabal,Spain,FWD,0.580401,0.378873,0.4950,1.00,0.033175,0.252492,0.252492,0.571429,0.533333,1.000000
149,Ousmane Dembélé,France,MID,0.539992,0.407246,0.4762,1.00,0.024038,0.263889,0.164931,0.571429,0.333333,1.000000
198,Erling Haaland,Norway,FWD,0.524094,0.306133,0.4717,0.75,0.027778,0.351852,0.316667,0.714286,0.600000,1.000000
127,Harry Kane,England,FWD,0.515050,0.282523,0.4167,0.75,0.022727,0.268362,0.241525,0.714286,0.600000,1.000000
185,Ismael Saibari,Morocco,MID,0.461903,0.306347,0.4049,0.50,0.007299,0.157025,0.078512,0.428571,0.200000,1.000000
350,Cristiano Ronaldo,Portugal,FWD,0.457495,0.248913,0.3846,0.75,0.026820,0.162393,0.189459,0.428571,0.466667,1.000000
49,Matheus Cunha,Brazil,FWD,0.430282,0.326276,0.4167,0.75,0.024272,0.242553,0.202128,0.428571,0.333333,1.000000


In [719]:
prob_assists = (
    0.15 * df["chance_created_per90"]
  + 0.20 * df["stat_CC"]
  + 0.15 * df["prob_scores"]
  + 0.15 * df["team_score_2_prob"]
  + 0.15 * df["recent_assists_last3"]
  + 0.10 * df["price"]
  + 0.15 * df["stat_AS"]
)

df["lambda_assist"] = -np.log(
    (1 - prob_assists).clip(lower=0.001)
)

df["expected_assists"] = df["lambda_assist"].clip(lower=0) * df["prob_plays_60min"]

In [720]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "stat_CC",
    "recent_cc_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "price",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,stat_CC,recent_cc_per90,team_score_2_prob,recent_assists_last3,price,prob_plays_60min,stat_AS
161,Michael Olise,France,MID,1.328938,1.249481,1.0,0.178899,0.7634,1.00,0.857143,0.845736,1.0
148,Kylian Mbappé,France,FWD,0.737536,0.219282,0.2,0.049808,0.7634,0.50,1.000000,0.897936,0.4
5,Lionel Messi,Argentina,FWD,0.663625,0.910125,0.8,0.216667,0.6452,0.00,0.928571,0.899661,0.0
321,Bruno Guimarães,Brazil,MID,0.657239,0.621794,0.6,0.150000,0.5714,0.75,0.471429,0.875022,0.8
257,Marc Cucurella,Spain,DEF,0.555977,0.574050,0.6,0.096296,0.5464,0.75,0.228571,0.898167,0.6
149,Ousmane Dembélé,France,MID,0.531818,0.267250,0.2,0.062500,0.7634,0.50,0.928571,0.801957,0.4
171,Roberto Alvarado,Mexico,FWD,0.520762,0.660071,0.8,0.156000,0.3425,0.50,0.257143,0.865252,0.6
285,Breel Embolo,Switzerland,FWD,0.509784,0.478686,0.6,0.050584,0.3390,0.50,0.571429,0.889168,0.4
48,Vinícius Júnior,Brazil,MID,0.450838,0.200769,0.2,0.049808,0.5714,0.25,0.928571,0.894919,0.2
140,Jude Bellingham,England,MID,0.409848,0.582994,0.6,0.166667,0.4049,0.25,0.685714,0.855402,0.2


In [721]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.4 * df["yc_per90"]
  + 0.25 * df["tackles_per90"]
  + 0.20 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.1 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = raw_yc * df["prob_plays_60min"]

In [722]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
209,Matías Galarza,Paraguay,MID,0.335370,0.255034,1.0,0.046980,0.234899,1.000000,0.909457,0.000000
111,Yasser Ibrahim,Egypt,DEF,0.285791,0.202128,1.0,0.018617,0.104895,0.894349,0.923769,0.000000
360,Andrés Cubas,Paraguay,MID,0.281007,0.097436,0.5,0.035897,0.150000,1.000000,0.927574,0.000000
216,Juan José Cáceres,Paraguay,DEF,0.263572,0.108883,0.5,0.051576,0.240741,1.000000,0.874355,0.000000
100,Marwan Attia,Egypt,MID,0.253704,0.110465,0.5,0.017442,0.059055,0.894349,0.898334,0.000000
212,Júnior Alonso,Paraguay,DEF,0.243877,0.126667,0.5,0.026667,0.142857,1.000000,0.806649,0.000000
211,Gustavo Gómez,Paraguay,DEF,0.232589,0.000000,0.0,0.010256,0.050000,1.000000,0.939509,0.000000
59,Casemiro,Brazil,MID,0.227225,0.262976,1.0,0.041522,0.225410,0.521910,0.826414,0.000000
218,Miguel Almirón,Paraguay,MID,0.226822,0.177570,0.5,0.032710,0.185185,1.000000,0.611949,0.728972
189,Patrick Berg,Norway,MID,0.225200,0.162393,0.5,0.017094,0.066667,0.830040,0.775912,0.000000


In [723]:
raw_pw = (
    0.10 * df["stat_PW"]                    
  + 0.3 * df["anytime_scorer_prob"]      
  + 0.10 * df["chance_created_per90"]     
  + 0.2 * df["stat_CC"]                  # recent form in 
  + 0.2 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price"] # quality proxy
)

df["prob_pen_won"] = 0.2 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [724]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
5,Lionel Messi,Argentina,FWD,0.074291,0.0,0.6173,0.910125,0.125000,0.6452,1.000000,0.899661
161,Michael Olise,France,MID,0.066213,0.0,0.3571,1.249481,0.162338,0.7634,0.909091,0.845736
148,Kylian Mbappé,France,FWD,0.055986,0.0,0.6536,0.219282,0.028490,0.7634,1.000000,0.897936
318,Lautaro Martínez,Argentina,FWD,0.042710,1.0,0.4587,0.299630,0.041152,0.6452,0.750000,0.697257
149,Ousmane Dembélé,France,MID,0.040338,0.0,0.4762,0.267250,0.034722,0.7634,1.000000,0.801957
48,Vinícius Júnior,Brazil,MID,0.038558,0.0,0.4717,0.200769,0.028490,0.5714,1.000000,0.894919
185,Ismael Saibari,Morocco,MID,0.034491,0.0,0.4049,0.375174,0.055096,0.5181,1.000000,0.915987
261,Mikel Oyarzabal,Spain,FWD,0.033136,0.0,0.4950,0.228857,0.033223,0.5464,1.000000,0.848337
39,Youri Tielemans,Belgium,MID,0.032822,1.0,0.2151,0.170977,0.025974,0.4878,0.727273,0.930263
285,Breel Embolo,Switzerland,FWD,0.032288,0.0,0.3175,0.478686,0.086455,0.3390,1.000000,0.889168


In [725]:
raw_cs = (
    0.90 * df["team_cs_prob"]            # strongest signal
  + 0.05 * df["stat_CS"]               # tournament history
  + 0.05 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [726]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
143,Dayot Upamecano,France,DEF,0.552690,0.5811,0.887167,0.50,0.285714,1.000000,0.020231
152,Mike Maignan,France,GK,0.548444,0.5811,0.880352,0.50,0.285714,1.000000,0.000000
10,Emiliano Martínez,Argentina,GK,0.536918,0.5479,0.942676,0.50,0.428571,1.000000,0.000000
19,Alexis Mac Allister,Argentina,MID,0.536313,0.5479,0.896554,0.75,0.285714,1.000000,0.036474
330,Jules Koundé,France,DEF,0.527611,0.5811,0.846911,0.50,0.285714,0.888889,0.032836
161,Michael Olise,France,MID,0.526879,0.5811,0.845736,0.50,0.285714,1.000000,0.012987
149,Ousmane Dembélé,France,MID,0.521462,0.5811,0.801957,0.75,0.142857,0.909091,0.010417
314,Lisandro Martínez,Argentina,DEF,0.501811,0.5479,0.865129,0.50,0.285714,1.000000,0.010000
20,Enzo Fernández,Argentina,MID,0.496180,0.5479,0.855420,0.50,0.285714,0.909091,0.020000
329,William Saliba,France,DEF,0.485502,0.5811,0.766993,0.50,0.142857,0.777778,0.007407


In [727]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.10 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.25 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.25 * df["opp_over_05_prob"]      # more defending -> more challenges
)

df["prob_red_card"] = 0.1 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [728]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
218,Miguel Almirón,Paraguay,MID,0.031198,0.14,0.728972,0.032710,0.226822,1.000000,0.657963
209,Matías Galarza,Paraguay,MID,0.029664,0.14,0.000000,0.046980,0.335370,1.000000,0.863946
360,Andrés Cubas,Paraguay,MID,0.028721,0.14,0.000000,0.035897,0.281007,1.000000,0.858149
211,Gustavo Gómez,Paraguay,DEF,0.027320,0.12,0.000000,0.010256,0.232589,1.000000,0.931775
216,Juan José Cáceres,Paraguay,DEF,0.026700,0.12,0.000000,0.051576,0.263572,1.000000,0.805304
111,Yasser Ibrahim,Egypt,DEF,0.025746,0.12,0.000000,0.018617,0.285791,0.894349,0.867291
227,Orlando Gill,Paraguay,GK,0.025189,0.04,0.000000,0.000000,0.198690,1.000000,0.845535
107,Mohamed Hany,Egypt,DEF,0.024783,0.12,0.000000,0.035897,0.218130,0.894349,0.902362
100,Marwan Attia,Egypt,MID,0.024482,0.14,0.000000,0.017442,0.253704,0.894349,0.837974
212,Júnior Alonso,Paraguay,DEF,0.023898,0.12,0.000000,0.026667,0.243877,1.000000,0.755859


In [729]:
position_og_modifier = {
    "GK": 0.05,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.5 * df["opp_over_05_prob"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [730]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
209,Matías Galarza,Paraguay,MID,0.017038,0.04,0.000000,1.000000,0.046980,0.142857,0.863946
211,Gustavo Gómez,Paraguay,DEF,0.016210,0.08,0.000000,1.000000,0.010256,0.714286,0.931775
216,Juan José Cáceres,Paraguay,DEF,0.015542,0.08,0.000000,1.000000,0.051576,0.571429,0.805304
227,Orlando Gill,Paraguay,GK,0.015537,0.05,0.000000,1.000000,0.000000,0.714286,0.845535
360,Andrés Cubas,Paraguay,MID,0.015458,0.04,0.000000,1.000000,0.035897,0.714286,0.858149
111,Yasser Ibrahim,Egypt,DEF,0.015111,0.08,0.000000,0.894349,0.018617,0.428571,0.867291
107,Mohamed Hany,Egypt,DEF,0.014825,0.08,0.005128,0.894349,0.035897,0.571429,0.902362
108,Ramy Rabia,Egypt,DEF,0.014496,0.08,0.000000,0.894349,0.014388,0.285714,0.785657
117,Mostafa Shobeir,Egypt,GK,0.014182,0.05,0.000000,0.894349,0.000000,0.571429,0.843895
100,Marwan Attia,Egypt,MID,0.014159,0.04,0.000000,0.894349,0.017442,0.428571,0.837974


In [731]:
position_pc_modifier = {
    "GK": 0.08,
    "FWD": 0.01,
    "MID": 0.05,
    "DEF": 0.14,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.20 * df["position_pc_modifier"]
  + 0.15 * df["penalty_conceded_rate"]
  + 0.15 * df["tackles_per90"]
  + 0.35 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.08 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [732]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
209,Matías Galarza,Paraguay,MID,0.018474,0.05,0.0,0.046980,0.335370,1.000000,0.863946
211,Gustavo Gómez,Paraguay,DEF,0.017461,0.14,0.0,0.010256,0.232589,1.000000,0.931775
111,Yasser Ibrahim,Egypt,DEF,0.017435,0.14,0.0,0.018617,0.285791,0.894349,0.867291
360,Andrés Cubas,Paraguay,MID,0.017425,0.05,0.0,0.035897,0.281007,1.000000,0.858149
216,Juan José Cáceres,Paraguay,DEF,0.017337,0.14,0.0,0.051576,0.263572,1.000000,0.805304
107,Mohamed Hany,Egypt,DEF,0.016297,0.14,0.0,0.035897,0.218130,0.894349,0.902362
227,Orlando Gill,Paraguay,GK,0.015558,0.08,0.0,0.000000,0.198690,1.000000,0.845535
212,Júnior Alonso,Paraguay,DEF,0.015364,0.14,0.0,0.026667,0.243877,1.000000,0.755859
100,Marwan Attia,Egypt,MID,0.015125,0.05,0.0,0.017442,0.253704,0.894349,0.837974
108,Ramy Rabia,Egypt,DEF,0.014287,0.14,0.0,0.014388,0.195257,0.894349,0.785657


In [733]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [734]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
211,Gustavo Gómez,Paraguay,DEF,0.939509,0.1448,1.932402,0.1448,0.279812,0.270354,0.174144,0.130889,1.815508,-1.012040
227,Orlando Gill,Paraguay,GK,0.924142,0.1448,1.932402,0.1448,0.279812,0.270354,0.174144,0.130889,1.785813,-0.995487
216,Juan José Cáceres,Paraguay,DEF,0.874355,0.1448,1.932402,0.1448,0.279812,0.270354,0.174144,0.130889,1.689605,-0.941857
212,Júnior Alonso,Paraguay,DEF,0.806649,0.1448,1.932402,0.1448,0.279812,0.270354,0.174144,0.130889,1.558769,-0.868923
107,Mohamed Hany,Egypt,DEF,0.936805,0.1910,1.655482,0.1910,0.316197,0.261729,0.144429,0.086644,1.550864,-0.792988
111,Yasser Ibrahim,Egypt,DEF,0.923769,0.1910,1.655482,0.1910,0.316197,0.261729,0.144429,0.086644,1.529283,-0.781954
117,Mostafa Shobeir,Egypt,GK,0.922371,0.1910,1.655482,0.1910,0.316197,0.261729,0.144429,0.086644,1.526968,-0.780770
108,Ramy Rabia,Egypt,DEF,0.858397,0.1910,1.655482,0.1910,0.316197,0.261729,0.144429,0.086644,1.421060,-0.726617
232,Nuno Mendes,Portugal,DEF,0.887945,0.2346,1.449873,0.2346,0.340140,0.246580,0.119170,0.059510,1.287408,-0.607775
236,Renato Veiga,Portugal,DEF,0.885709,0.2346,1.449873,0.2346,0.340140,0.246580,0.119170,0.059510,1.284166,-0.606244


In [735]:
raw_save = (
    0.35 * df["expected_goals_conceded_norm"]  # opportunity
  + 0.40 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0

In [736]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "expected_goals_conceded_norm",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,expected_goals_conceded_norm
227,Orlando Gill,Paraguay,3.018138,1.932402,0.876923,0.960000,0.000000,1.000000
346,Diogo Costa,Portugal,2.233683,1.449873,0.700000,0.866667,0.200000,0.652750
288,Gregor Kobel,Switzerland,1.772131,1.235808,0.650000,0.666667,0.171429,0.498698
117,Mostafa Shobeir,Egypt,1.766566,1.655482,0.461538,0.420000,0.000000,0.800715
340,Thibaut Courtois,Belgium,1.399126,1.297917,0.415385,0.420000,0.200000,0.543395
56,Alisson Becker,Brazil,1.345255,1.046969,0.550000,0.600000,0.214286,0.362801
202,Ørjan Nyland,Norway,1.331678,1.527858,0.400000,0.600000,0.100000,0.708871
70,Maxime Crépeau,Canada,0.989761,1.413049,0.250000,0.200000,0.071429,0.626249
311,Matt Freese,USA,0.967453,1.314532,0.333333,0.500000,0.100000,0.555352
176,Raúl Rangel,Mexico,0.797179,1.133204,0.310345,0.279070,0.057143,0.424859


In [737]:
raw_pen_save = (
    0.55 * df["price"]                    
  + 0.30 * df["expected_goals_conceded_norm"]
  + 0.15 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [738]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
117,Mostafa Shobeir,Egypt,0.019296,0.000000,1.655482,0.894349,0.922371,1.0
227,Orlando Gill,Paraguay,0.016373,0.000000,1.932402,1.000000,0.924142,0.0
346,Diogo Costa,Portugal,0.015774,0.200000,1.449873,0.784669,0.880352,0.0
340,Thibaut Courtois,Belgium,0.015528,0.200000,1.297917,0.700513,0.924142,0.0
152,Mike Maignan,France,0.014640,0.214286,0.542832,0.000000,0.880352,1.0
288,Gregor Kobel,Switzerland,0.013947,0.171429,1.235808,0.661417,0.880352,0.0
70,Maxime Crépeau,Canada,0.013473,0.071429,1.413049,0.770053,0.880352,0.0
56,Alisson Becker,Brazil,0.013460,0.214286,1.046969,0.521910,0.880352,0.0
265,Unai Simón,Spain,0.013429,0.214286,1.041854,0.521910,0.880352,0.0
202,Ørjan Nyland,Norway,0.012650,0.100000,1.527858,0.830040,0.760983,0.0


In [739]:
raw_tackles = (
    0.50 * df["tackles_per90"]
  + 0.15 * df["recent_tackles_per90"]
  + 0.15 * df["expected_goals_conceded_norm"]
  + 0.20 * df["match_over_25_prob"]
)

df["lambda_tackles"] = -np.log(
    (1 - raw_tackles).clip(lower=0.001)
)

df["expected_tackles"] = (
    df["lambda_tackles"].clip(lower=0)
    * df["prob_plays_60min"]
)

df.loc[df["position"] != "MID", "expected_tackles"] = 0.0

In [740]:
cols = [
    "name",
    "team",
    "expected_tackle_points",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackle_points", ascending=False) \
    .head(30)

,name,team,expected_tackle_points,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
209,Matías Galarza,Paraguay,0.233753,0.046980,0.234899,1.932402,0.5718,0.185714,0.909457
360,Andrés Cubas,Paraguay,0.219508,0.035897,0.150000,1.932402,0.5718,0.171429,0.927574
229,Julio Enciso,Paraguay,0.170678,0.012270,0.042373,1.932402,0.5718,0.442857,0.821783
100,Marwan Attia,Egypt,0.151411,0.017442,0.059055,1.655482,0.4816,0.114286,0.898334
218,Miguel Almirón,Paraguay,0.147274,0.032710,0.185185,1.932402,0.5718,0.357143,0.611949
249,João Neves,Portugal,0.145433,0.029900,0.189573,1.449873,0.5141,0.428571,0.820629
298,Weston McKennie,USA,0.143209,0.022599,0.151515,1.314532,0.5365,0.371429,0.890555
103,Emam Ashour,Egypt,0.142975,0.009346,0.040000,1.655482,0.4816,0.157143,0.878122
228,Damián Bobadilla,Paraguay,0.142137,0.036842,0.241379,1.932402,0.5718,0.285714,0.563233
359,Mohamed Salah,Egypt,0.140672,0.002959,0.000000,1.655482,0.4816,0.928571,0.904995


In [741]:
raw_cc = (
    0.25 * df["recent_cc_per90"]
  + 0.25 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.15 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_cc"] = -np.log(
    (1 - raw_cc).clip(lower=0.001)
)

df["expected_chances_created"] = (
    df["lambda_cc"].clip(lower=0)
    * df["prob_plays_60min"]
)

df.loc[df["position"] != "MID", "expected_chances_created"] = 0.0

In [742]:
cols = [
    "name",
    "team",
    "position",
    "expected_cc_points",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_cc_points", ascending=False).head(30)

,name,team,position,expected_cc_points,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
161,Michael Olise,France,MID,0.778191,0.162338,0.178899,1.0,0.5718,0.3571,0.857143,0.845736
140,Jude Bellingham,England,MID,0.455714,0.095541,0.166667,0.6,0.4076,0.2564,0.685714,0.855402
321,Bruno Guimarães,Brazil,MID,0.449912,0.088235,0.150000,0.6,0.5352,0.1538,0.471429,0.875022
48,Vinícius Júnior,Brazil,MID,0.404543,0.028490,0.049808,0.2,0.5352,0.4717,0.928571,0.894919
250,Bruno Fernandes,Portugal,MID,0.384913,0.060060,0.106996,0.4,0.5141,0.2151,0.714286,0.854701
149,Ousmane Dembélé,France,MID,0.382573,0.034722,0.062500,0.2,0.5718,0.4762,0.928571,0.801957
185,Ismael Saibari,Morocco,MID,0.381955,0.055096,0.094891,0.4,0.4373,0.4049,0.471429,0.915987
344,Leandro Trossard,Belgium,MID,0.381914,0.055402,0.095941,0.4,0.5365,0.2740,0.442857,0.905769
189,Patrick Berg,Norway,MID,0.373049,0.128205,0.173333,0.6,0.5352,0.1176,0.300000,0.775912
357,Mostafa Zico,Egypt,MID,0.340402,0.101695,0.178082,0.6,0.4816,0.1852,0.100000,0.776133


In [743]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.20 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_sot"] = -np.log(
    (1 - raw_sot).clip(lower=0.001)
)

df["expected_shots_on_target"] = (
    df["lambda_sot"].clip(lower=0)
    * df["prob_plays_60min"]
)

df.loc[df["position"] != "FWD", "expected_shots_on_target"] = 0.0

In [744]:
cols = [
    "name",
    "team",
    "position",
    "expected_sot_points",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_sot_points", ascending=False).head(30)

,name,team,position,expected_sot_points,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
5,Lionel Messi,Argentina,FWD,0.904992,0.445312,0.415625,1.000000,0.4816,0.6173,0.928571,0.899661
148,Kylian Mbappé,France,FWD,0.863703,0.351852,0.324786,0.857143,0.5718,0.6536,1.000000,0.897936
198,Erling Haaland,Norway,FWD,0.640658,0.316667,0.351852,0.714286,0.5352,0.4717,1.000000,0.807380
127,Harry Kane,England,FWD,0.588515,0.241525,0.268362,0.714286,0.4076,0.4167,1.000000,0.901582
261,Mikel Oyarzabal,Spain,FWD,0.525946,0.252492,0.252492,0.571429,0.5141,0.4950,0.657143,0.848337
350,Cristiano Ronaldo,Portugal,FWD,0.479638,0.189459,0.162393,0.428571,0.5141,0.3846,0.928571,0.891340
49,Matheus Cunha,Brazil,FWD,0.388650,0.202128,0.242553,0.428571,0.5352,0.4167,0.542857,0.792376
65,Jonathan David,Canada,FWD,0.338614,0.200906,0.172205,0.428571,0.4373,0.2299,0.500000,0.898334
31,Romelu Lukaku,Belgium,FWD,0.292613,0.107345,0.214689,0.285714,0.5365,0.4587,0.557143,0.719496
285,Breel Embolo,Switzerland,FWD,0.291908,0.109510,0.109510,0.285714,0.4306,0.3175,0.571429,0.889168


In [745]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 2

In [746]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_qualification_points", ascending=False).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
332,Jean-Philippe Mateta,France,FWD,1.886792,0.943396,0.428571,0.181230
329,William Saliba,France,DEF,1.886792,0.943396,0.257143,0.766993
330,Jules Koundé,France,DEF,1.886792,0.943396,0.271429,0.846911
331,Bradley Barcola,France,MID,1.886792,0.943396,0.642857,0.698293
153,Brice Samba,France,GK,1.886792,0.943396,0.142857,0.112047
157,Manu Koné,France,MID,1.886792,0.943396,0.371429,0.435794
155,Adrien Rabiot,France,MID,1.886792,0.943396,0.414286,0.725280
141,Theo Hernández,France,DEF,1.886792,0.943396,0.214286,0.418340
328,Robin Risser,France,GK,1.886792,0.943396,0.000000,0.123467
152,Mike Maignan,France,GK,1.886792,0.943396,0.214286,0.880352


In [747]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 414 rows x 229 cols


In [748]:
BUDGET = 105.0
MAX_PER_COUNTRY = 4
SQUAD_SIZE = 15
XI_SIZE = 11
BENCH_WEIGHT = 0.6  

def compute_total_expected_points(df):
    df = df.copy()

    cs_value = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
    goal_value = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}

    df["goal_pts_value"] = df["position"].map(goal_value)
    df["cs_pts_value"] = df["position"].map(cs_value)

    # appearance: 1pt for playing any, +1 if 60+
    e_appearance = df["prob_plays_any"] * 1 + df["prob_plays_60min"] * 1

    # goal
    e_goal = df["prob_scores"] * df["goal_pts_value"]

    # assist
    e_assist = df["expected_assists"] * 3

    # clean sheet: gated by 60min, value depends on position
    e_cs = df["prob_clean_sheet"] * df["cs_pts_value"]

    # goals conceded penalty: GK and DEF only
    e_gc = df["expected_gc_penalty"].copy()
    e_gc[~df["position"].isin(["GK", "DEF"])] = 0.0

    # saves bonus: GK only, every 3 saves = +1
    e_saves = (df["expected_saves"] / 3).copy()
    e_saves[df["position"] != "GK"] = 0.0

    # penalty save: GK only
    e_pen_save = (df["prob_penalty_save"] * 3).copy()
    e_pen_save[df["position"] != "GK"] = 0.0

    # tackles bonus: MID only, every 3 tackles = +1
    e_tackles = df["expected_tackle_points"].copy()
    e_tackles[df["position"] != "MID"] = 0.0

    # big chances created: MID only, every 2 = +1
    e_cc = df["expected_cc_points"].copy()
    e_cc[df["position"] != "MID"] = 0.0

    # shots on target: FWD only, every 2 = +1
    e_sot = df["expected_sot_points"].copy()
    e_sot[df["position"] != "FWD"] = 0.0

    # yellow card
    e_yc = df["prob_yellow_card"] * -1

    # red card
    e_rc = df["prob_red_card"] * -2

    # own goal
    e_og = df["prob_own_goal"] * -2

    # penalty won
    e_pw = df["prob_pen_won"] * 2

    # penalty committed
    e_pc = df["prob_penalty_committed"] * -1

    # qualification bonus: +2 per player in XI who advances, requires playing
    # captain's quali bonus is NOT doubled per rules
    e_quali = df["expected_qualification_points"] * df["prob_plays_any"]

    # scouting bonus: +2 if differential==1 AND player scores >4 base pts
    # model: use Poisson confidence interval on expected base points
    # base pts = everything except scouting and captain multiplier
    base_ep = (
        e_appearance + e_goal + e_assist + e_cs + e_gc +
        e_saves + e_pen_save + e_tackles + e_cc + e_sot +
        e_yc + e_rc + e_og + e_pw + e_pc + e_quali
    )

    # Poisson 90% CI: lower = ppf(0.05, mu), upper = ppf(0.95, mu)
    mu = base_ep.clip(lower=0.01)
    lower_ci = pd.Series(poisson.ppf(0.05, mu), index=df.index)
    upper_ci = pd.Series(poisson.ppf(0.95, mu), index=df.index)

    # if lower CI > 4: full +2 expected
    # if upper CI < 4: 0
    # if CI straddles 4: scale by P(X>4 | mu) * 2
    p_exceeds_4 = pd.Series(1 - poisson.cdf(4, mu), index=df.index)

    scouting_ep = pd.Series(0.0, index=df.index)
    eligible = df["differential"] == 1
    full_bonus = eligible & (lower_ci > 4)
    no_bonus = eligible & (upper_ci < 4)
    partial = eligible & ~full_bonus & ~no_bonus

    scouting_ep[full_bonus] = 2.0
    scouting_ep[no_bonus] = 0.0
    scouting_ep[partial] = p_exceeds_4[partial] * 2.0

    df["e_appearance"] = e_appearance
    df["e_goal"] = e_goal
    df["e_assist"] = e_assist
    df["e_cs"] = e_cs
    df["e_gc"] = e_gc
    df["e_saves"] = e_saves
    df["e_pen_save"] = e_pen_save
    df["e_tackles"] = e_tackles
    df["e_cc"] = e_cc
    df["e_sot"] = e_sot
    df["e_yc"] = e_yc
    df["e_rc"] = e_rc
    df["e_og"] = e_og
    df["e_pw"] = e_pw
    df["e_pc"] = e_pc
    df["e_quali"] = e_quali
    df["e_scouting"] = scouting_ep
    df["expected_points"] = base_ep + scouting_ep

    return df

In [749]:

def optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):
    idx = df.index.tolist()

    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi", idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")    
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("fantasy", LpMaximize)

    
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
        for i in idx
    )

    # squad = 15, xi = 11, bench = 4
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    # budget
    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    # squad composition: 2 GK, 5 DEF, 5 MID, 3 FWD
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # country limit
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    # XI formation: 1 GK starts, valid outfield formation
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # captain: exactly 1, must be in XI
    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "squad": df.loc[in_squad].copy(),
        "xi": df.loc[in_xi].copy(),
        "bench": df.loc[on_bench].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
        "cost": df.loc[in_squad, "price_raw"].sum(),
    }

In [750]:
def optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY):
    # best possible 11 ignoring bench constraint, just 11 players
    idx = df.index.tolist()

    s = LpVariable.dicts("xi", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("ideal_xi", LpMaximize)

    # optimize_ideal_xi objective — NO bench term
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    model += lpSum(s[i] for i in idx) == 11

    # no budget constraint for ideal XI comparison
    # country limit still applies
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(s[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        return None

    in_xi = [i for i in idx if value(s[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "xi": df.loc[in_xi].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
    }


def print_team(result, ideal=None):
    pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
    cap = result["captain"]

    xi = result["xi"].copy()
    xi["_ord"] = xi["position"].map(pos_order)
    xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])

    bench = result["bench"].copy()
    bench["_ord"] = bench["position"].map(pos_order)
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    print(f"\nExpected points: {result['total_ep']:.2f}   Cost: ${result['cost']:.1f}M\n")

    print("Starting XI")
    for i, row in xi.iterrows():
        tag = " [C]" if i == cap else ""
        scout = " [SCOUT]" if row.get("differential", 0) == 1 else ""
        print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    print("\nBench")
    for rank, (i, row) in enumerate(bench.iterrows(), 1):
        print(f"  [{rank}] {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}")

    print(f"\nCountry breakdown: {dict(result['squad']['team'].value_counts())}")

    if ideal:
        ideal_xi = ideal["xi"].copy()
        ideal_xi["_ord"] = ideal_xi["position"].map(pos_order)
        ideal_xi = ideal_xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        ideal_cap = ideal["captain"]

        print(f"\nIdeal XI (no budget constraint, 11 only, EP: {ideal['total_ep']:.2f})")
        for i, row in ideal_xi.iterrows():
            tag = " [C]" if i == ideal_cap else ""
            in_squad = i in result["squad"].index
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")
            
if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)

    df = compute_total_expected_points(df)

    result = optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)

    print_team(result, ideal)


Expected points: 130.66   Cost: $104.7M

Starting XI
  GK   Emiliano Martínez            Argentina              $5.0  EP:7.96
  DEF  Dayot Upamecano              France                 $5.3  EP:7.72
  DEF  Achraf Hakimi                Morocco                $6.0  EP:7.65
  DEF  Lisandro Martínez            Argentina              $4.6  EP:7.34
  MID  Michael Olise                France                 $9.5  EP:11.53
  MID  Ousmane Dembélé              France                 $10.0  EP:9.76
  MID  Vinícius Júnior              Brazil                 $10.0  EP:9.34
  MID  Alexis Mac Allister          Argentina              $6.6  EP:8.48 [SCOUT]
  MID  Ismael Saibari               Morocco                $6.8  EP:8.22
  FWD  Kylian Mbappé                France                 $10.5  EP:12.23 [C]
  FWD  Lionel Messi                 Argentina              $10.0  EP:11.47

Bench
  [1] GK   Yassine Bounou               Morocco                $4.7  EP:6.89
  [2] DEF  Dávinson Sánchez             

In [751]:
def optimize_with_transfers(df, current_team_names, free_transfers=4, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 4} penalised at -3 each)" if result['n_transfers'] > 4 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Jordan Pickford", "Lisandro Martínez", "Sergiño Dest", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Jamal Musiala", "Vinícius Júnior", "Christian Pulisic", "Kylian Mbappé", "Harry Kane",
        "Camilo Vargas", "Facundo Medina", "Lionel Messi", "Johan Manzambi"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=4, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

    print("\nIdeal XI for comparison:")
    if ideal:
        pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
        xi = ideal["xi"].copy()
        xi["_ord"] = xi["position"].map(pos_order)
        xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        cap = ideal["captain"]
        for i, row in xi.iterrows():
            tag = " [C]" if i == cap else ""
            in_squad = i in result["squad"].index if result else False
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")

Eliminated players: ['Jamal Musiala']

Transfers made: 4 (all free)
Point penalty: -0

OUT:
  Jordan Pickford
  Christian Pulisic
  Sergiño Dest
  Jamal Musiala
IN:
  Michael Olise
  Ismael Saibari
  Emiliano Martínez
  Dávinson Sánchez

Expected points: 124.18   Cost: $104.9M

Starting XI
  GK   Emiliano Martínez            Argentina              $5.0  EP:7.96
  DEF  Lisandro Martínez            Argentina              $4.6  EP:7.34
  DEF  Marc Cucurella               Spain                  $5.1  EP:6.91
  DEF  Dávinson Sánchez             Colombia               $4.3  EP:6.66 [SCOUT]
  MID  Michael Olise                France                 $9.5  EP:11.53
  MID  Ousmane Dembélé              France                 $10.0  EP:9.76
  MID  Vinícius Júnior              Brazil                 $10.0  EP:9.34
  MID  Ismael Saibari               Morocco                $6.8  EP:8.22
  FWD  Kylian Mbappé                France                 $10.5  EP:12.23 [C]
  FWD  Lionel Messi                

In [752]:
for pos in ["GK", "DEF", "MID", "FWD"]:
    print(f"\n--- {pos} ---")
    print(df[df["position"] == pos].nlargest(10, "expected_points")[["name", "team", "price_raw", "expected_points"]].to_string(index=False))


--- GK ---
             name        team  price_raw  expected_points
Emiliano Martínez   Argentina        5.0         7.956293
     Mike Maignan      France        5.0         7.176301
   Yassine Bounou     Morocco        4.7         6.891817
    Camilo Vargas    Colombia        4.3         6.041175
   Alisson Becker      Brazil        5.0         5.119327
       Unai Simón       Spain        5.0         5.099955
     Gregor Kobel Switzerland        4.7         4.870678
  Jordan Pickford     England        4.8         4.734632
 Thibaut Courtois     Belgium        4.9         4.291078
      Matt Freese         USA        4.2         4.152908

--- DEF ---
             name      team  price_raw  expected_points
  Dayot Upamecano    France        5.3         7.716034
    Achraf Hakimi   Morocco        6.0         7.645619
     Jules Koundé    France        5.4         7.477421
Lisandro Martínez Argentina        4.6         7.340416
    Nahuel Molina Argentina        4.4         7.140207
 

In [753]:
player1 = "Harry Kane"
player2 = "Breel Embolo"

cols = [
    "name", "team", "position", "price_raw", "expected_points",
    "e_appearance", "e_goal", "e_assist", "e_cs", "e_gc",
    "e_saves", "e_pen_save", "e_tackles", "e_cc", "e_sot",
    "e_yc", "e_rc", "e_og", "e_pw", "e_pc",
    "e_quali", "e_scouting"
]

comparison = df.loc[
    df["name"].isin([player1, player2]),
    cols
].set_index("name").T

print(comparison)

name            Harry Kane Breel Embolo
team               England  Switzerland
position               FWD          FWD
price_raw             10.5          7.5
expected_points   7.292743     7.866649
e_appearance      1.820453     1.802803
e_goal            3.090299     2.013058
e_assist          0.790555     1.529352
e_cs                   0.0          0.0
e_gc                   0.0          0.0
e_saves                0.0          0.0
e_pen_save             0.0          0.0
e_tackles              0.0          0.0
e_cc                   0.0          0.0
e_sot             0.588515     0.291908
e_yc             -0.115285    -0.149384
e_rc             -0.026975    -0.033944
e_og             -0.014313    -0.020781
e_pw              0.054767     0.064575
e_pc             -0.009058    -0.010709
e_quali           1.113784      0.86192
e_scouting             0.0     1.517851
